# 00 — Explorar

Notebook de **solo lectura**: no escribe nada en `data/`, no reprocesa nada. Arma vistas
DuckDB sobre las capas del pipeline que ya existan en disco y deja unas consultas de ejemplo para arrancar un
análisis exploratorio sin tener que reescribir el boilerplate de lectura cada vez.

In [ ]:
import sys
from pathlib import Path

import duckdb


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

DATA = REPO / "data"
META = DATA / "meta"
con = duckdb.connect()
print(f"REPO = {REPO}")

## 1. Vistas sobre las capas del pipeline

`union_by_name=true` en todas -- el esquema de columnas cambia entre eras (ej. el
quiebre de 2023T4, ver `docs/decisiones_metodologicas.md` D2), así que no se puede
asumir que todos los Parquet de una capa comparten exactamente las mismas columnas.

In [ ]:
def registrar_vista_capa(nombre_vista: str, carpeta: Path) -> bool:
    """Crea `nombre_vista` sobre `carpeta/**/*.parquet` si hay al menos un Parquet
    ahí. Si la capa no existe todavía (o existe pero está vacía), no crea la vista
    y lo informa -- no rompe el notebook."""
    if not carpeta.exists():
        print(f"[ausente] {nombre_vista}: {carpeta} no existe todavía.")
        return False
    if not any(carpeta.rglob("*.parquet")):
        print(f"[vacío] {nombre_vista}: {carpeta} existe pero no tiene Parquet todavía.")
        return False
    glob = str(carpeta / "**" / "*.parquet")
    con.sql(f"CREATE OR REPLACE VIEW {nombre_vista} AS SELECT * FROM read_parquet(\'{glob}\', union_by_name=true)")
    n_filas = con.sql(f"SELECT count(*) FROM {nombre_vista}").fetchone()[0]
    print(f"[ok] {nombre_vista}: {n_filas:,} filas -- {carpeta}")
    return True


CAPAS = {
    "armonizado_hogar": DATA / "01_armonizado" / "hogar",
    "armonizado_individual": DATA / "01_armonizado" / "individual",
    "nucleo_hogar": DATA / "02_nucleo" / "hogar",
    "nucleo_individual": DATA / "02_nucleo" / "individual",
    "panel_pares_h1": DATA / "03_panel" / "pares_h1",
    "panel_pares_h4": DATA / "03_panel" / "pares_h4",
}

vistas_disponibles = {nombre: registrar_vista_capa(nombre, carpeta) for nombre, carpeta in CAPAS.items()}

## 2. Consultas de ejemplo

### 2.1 Trimestres disponibles por capa

In [ ]:
for nombre, disponible in vistas_disponibles.items():
    if not disponible:
        continue
    fila = con.sql(f"SELECT count(DISTINCT (ANO4, TRIMESTRE)) AS n_trimestres, min(ANO4) AS desde, max(ANO4) AS hasta FROM {nombre}").df()
    print(nombre, "->", fila.to_dict("records")[0])

### 2.2 Conteo de filas por trimestre (una capa puntual)

In [ ]:
if vistas_disponibles.get("nucleo_hogar"):
    con.sql("""
        SELECT ANO4, TRIMESTRE, count(*) AS n_hogares
        FROM nucleo_hogar
        GROUP BY ANO4, TRIMESTRE
        ORDER BY ANO4, TRIMESTRE
    """).df().tail(10)
else:
    print("nucleo_hogar no disponible.")

### 2.3 Columnas de una capa puntual

In [ ]:
if vistas_disponibles.get("nucleo_hogar"):
    display(con.sql("DESCRIBE SELECT * FROM nucleo_hogar").df())
else:
    print("nucleo_hogar no disponible.")

### 2.4 Cruce hogar + individual de un trimestre puntual

In [ ]:
ANIO_EJEMPLO, TRIMESTRE_EJEMPLO = 2023, 1

if vistas_disponibles.get("nucleo_hogar") and vistas_disponibles.get("nucleo_individual"):
    cruce = con.sql(f"""
        SELECT i.*, h.ingreso_no_declarado, h.ITF, h.IPCF
        FROM nucleo_individual i
        JOIN nucleo_hogar h
          ON i.CODUSU = h.CODUSU AND i.NRO_HOGAR = h.NRO_HOGAR
         AND i.ANO4 = h.ANO4 AND i.TRIMESTRE = h.TRIMESTRE
        WHERE i.ANO4 = {ANIO_EJEMPLO} AND i.TRIMESTRE = {TRIMESTRE_EJEMPLO}
    """).df()
    print(f"{len(cruce):,} personas en {ANIO_EJEMPLO}T{TRIMESTRE_EJEMPLO}")
    cruce.head()
else:
    print("nucleo_hogar / nucleo_individual no disponibles.")

### 2.5 Consulta directa sobre `data/meta/*.csv`

DuckDB puede leer CSV sin pasar por Parquet -- útil para los artefactos de diagnóstico
del pipeline (`flags_trimestre.csv`, `corridas.csv`), que son chicos y versionados.

In [ ]:
flags_path = META / "flags_trimestre.csv"
corridas_path = META / "corridas.csv"

if flags_path.exists():
    print("--- trimestres con aglomerados faltantes (flags_trimestre.csv) ---")
    display(con.sql(f"""
        SELECT * FROM read_csv_auto('{flags_path}')
        WHERE aglomerados_faltantes IS NOT NULL
        ORDER BY anio, trimestre
    """).df())
else:
    print(f"{flags_path} no existe todavía (correr notebook 01).")

if corridas_path.exists():
    print("\n--- últimas corridas registradas (corridas.csv) ---")
    display(con.sql(f"SELECT * FROM read_csv_auto('{corridas_path}') ORDER BY fecha_hora DESC LIMIT 10").df())
else:
    print(f"{corridas_path} no existe todavía.")